<a href="https://colab.research.google.com/github/IzanPereira/N-cleo-de-IA---para-estudo-do-datalab/blob/main/Detec%C3%A7%C3%A3o_de_Fraude_Kaggle.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Instalando as bibliotecas que serão utilizadas

In [ ]:
!pip install kaggle
!pip install kagglehub
!pip install polars

Entrando com API TOKEN

In [ ]:
!export KAGGLE_API_TOKEN=KGAT_1d86fe9e94e7c0ed77c696db6e47f4dc

Acessando via TOKEN

In [ ]:
mkdir -p ~/.kaggle && echo KGAT_1d86fe9e94e7c0ed77c696db6e47f4dc > ~/.kaggle/access_token && chmod 600 ~/.kaggle/access_token

Lista de competições

In [ ]:
!kaggle competitions list

ref                                                                           deadline             category         reward  teamCount  userHasEntered  
----------------------------------------------------------------------------  -------------------  --------  -------------  ---------  --------------  
https://www.kaggle.com/competitions/passenger-screening-algorithm-challenge   2017-12-15 23:59:00  Featured  1,500,000 Usd        518           False  
https://www.kaggle.com/competitions/zillow-prize-1                            2018-01-10 15:59:00  Featured  1,200,000 Usd       3770           False  
https://www.kaggle.com/competitions/data-science-bowl-2017                    2017-04-12 23:59:00  Featured  1,000,000 Usd       1972           False  
https://www.kaggle.com/competitions/vesuvius-challenge-ink-detection          2023-06-14 23:59:00  Featured  1,000,000 Usd       1249           False  
https://www.kaggle.com/competitions/arc-prize-2026-arc-agi-3                  2026-11-02

Buscando os comandos que podem ser utilizados

In [ ]:
!kaggle --help

usage: kaggle [-h] [-v] [-W]
              {competitions,c,datasets,d,kernels,k,models,m,files,f,benchmarks,b,config,auth}
              ...

options:
  -h, --help            show this help message and exit
  -v, --version         Print the Kaggle CLI version
  -W, --no-warn         Disable out-of-date API version warning

commands:
  {competitions,c,datasets,d,kernels,k,models,m,files,f,benchmarks,b,config,auth}
                        Use one of:
                        competitions {list, files, download, submit, submissions, leaderboard, episodes, replay, logs, pages}
                        datasets {list, files, download, create, version, init, metadata, status, delete}
                        kernels {list, files, get, init, push, pull, output, status, logs, update, delete}
                        models {instances, i, variations, v, get, list, init, create, delete, update}
                        models variations {versions, v, get, files, list, init, create, delete, update}
  

Baixando a pasta zipada

In [ ]:
!kaggle competitions download -c ieee-fraud-detection



ieee-fraud-detection.zip: Skipping, found more recently modified local copy (use --force to force download)


Descompactando

In [ ]:
!unzip ieee-fraud-detection.zip

Archive:  ieee-fraud-detection.zip
replace sample_submission.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [ ]:
import polars as pl
import polars.selectors as cs

In [ ]:
transtrain = pl.read_csv('train_transaction.csv')
identtrain = pl.read_csv('train_identity.csv')

In [ ]:
transtrain.schema



14 categoricas em transaction

In [ ]:
identtrain.schema


17 categorica em identity

Contei na mão quantas eram categorias, dai, pensei não é possível, ter que fazer isso, e pesquisei sobre, dai descobri a forma abaixo usando o polars.selectors (acho que posso chamar de biblioteca ou função) tenho dúvida nisso, dai na hora da categorica, tinha duas formas, usei uma deu zero dai usei a outra.

In [ ]:
len(transtrain.select(cs.numeric()).columns)

In [ ]:
len(identtrain.select(cs.numeric()).columns)

In [ ]:
len(transtrain.select(cs.string()).columns)

In [ ]:
len(identtrain.select(cs.string()).columns)

In [ ]:
TransIdent = transtrain.join(identtrain, on='TransactionID')

In [ ]:
print(TransIdent)

In [ ]:
TransIdent.null_count()

Não encontrei nenhum dados faltante

In [ ]:
data = TransIdent.select(
    'TransactionDT',
    'TransactionAmt',
    'ProductCD',
    'card1',
    'DeviceType',
    'isFraud'
)

Nesse eu fiquei na dúvida em .get_column ou .select fiz com um depois fiz com o outro, dai perguntei ao chate a diferença, get_column é só uma coluna

In [ ]:
data.describe()


In [ ]:
data.filter(pl.col('isFraud') == 1)

In [ ]:
propor = transtrain.select(
    pl.col("isFraud").sum().alias("qtd_fraudes"),pl.col("isFraud").count().alias("total_transacoes"),
     (pl.col("isFraud").sum() / pl.col("isFraud").count()).alias("proporcao_formula"))

In [ ]:
print(propor)

Confesso que essa foi bem desafiadora no quesito do que pra quê, a formula é tranquila, dificil é saber o que usar e como.

In [ ]:
Produto = data.group_by('ProductCD').agg([pl.sum('TransactionAmt'), pl.mean('isFraud')])

In [ ]:
print(Produto)

In [ ]:
Prod20 = data.filter(pl.col("isFraud") == 1).sort("TransactionAmt", descending=True)

In [ ]:
print(Prod20.head(20))

In [ ]:
data = data.with_columns(
    pl.col("TransactionAmt").log().alias("Amt_log"),
    ((pl.col("TransactionDT") // 3600) % 24).alias("Hour")
)

Nessa ((pl.col("TransactionDT") // 3600) % 24).alias("Hour") tive muita dificuldade saber o que deveria ter dai pedi ajuda para determinar e saber o que era, não achei algo que falava disso

In [ ]:
data = data.with_columns(
    pl.when(pl.col("TransactionAmt") < 50)
    .then(pl.lit("Baixo"))
    .when(pl.col("TransactionAmt") < 200)
    .then(pl.lit("Médio"))
    .otherwise(pl.lit("Alto"))
    .alias("Amt_category")
)

In [ ]:
print(data.head(10))

In [ ]:
data.group_by("DeviceType").agg(
    pl.col("isFraud").count().alias("numero_transacoes"),
    pl.col("isFraud").mean().alias("taxa_fraude"),
    pl.col("TransactionAmt").mean().alias("valor_medio")
).sort("taxa_fraude", descending=True)